**公开阅读版：** 不包含微博数据集，也不保留运行输出。课堂数据由教师另行提供，放在 `tutorials/data/posts_two_weeks.parquet`。Cookie和采集结果只保留在自己电脑。

# Big Data with DuckDB

**第一次课第三节的选讲补充。** 用中文说明、完整代码、实际输出与结果解读，
回答一个问题：把四川的用户行为分析扩展到全国，怎样减少读取、内存和中间数据？
本教程独立初始化，可从头运行；沿用同一份真实数据，不需要额外数据集。

**唯一原始数据：** `tutorials/data/posts_two_weeks.parquet`。
全国窗口为 2022 年 8 月 12–25 日，共 1,254,689 条采集记录、20 列，
Parquet 约 103 MiB；全国可用记录 1,149,693 条。
前面教程使用其中四川的 54,609 条可用记录。这里查询全国，也筛选四川核对结果。
“全国”表示这个文件内的全国城市标签记录，不表示全平台覆盖或完整用户历史。

### 讲解路线

- [1．文件大小与内存占用](#size)：3 分钟
- [2．连接 DuckDB，直接查询 Parquet](#connect)：3 分钟
- [3．SQL 入门：筛选与全国城市统计](#sql)：5 分钟
- [4．用户数、关键词比例与 Pandas 核对](#metrics)：6 分钟
- [5．汇总后绘图](#plot)：3 分钟
- [6．查询计划：少读列、提前筛选](#plan)：4 分钟
- [7．多核与内存：数据继续增大怎么办？](#resources)：选讲 8–10 分钟
- [附录：DuckDB Spatial 与第二次课](#spatial)：选讲 3 分钟
- [速查与参考资料](#reference)

基础路线约 25 分钟；完整路线约 35–40 分钟。例子已经提供完整答案，可现场修改省编码或关键词观察结果。
除可选整表内存演示外，Pandas 只接收统计结果和四川所需的三列。
图直接嵌在 Notebook；没有前端资源导出步骤。

### 环境准备

在 VS Code 打开整个课程项目，右上角选择课程 Conda 内核。
`requirements/environment.yml` 已声明 DuckDB 1.4 LTS 的 Python 包。
已有环境可在**终端**安装：

```sh
conda install -n user-behavior-class -c conda-forge 'python-duckdb>=1.4,<1.5'
```

如果使用另一个 Conda 环境，把环境名替换成自己的环境名。安装后重启 Notebook 内核。
Python 原生接口可以直接在 Notebook 中使用；`%%sql` 属于可选的 JupySQL 接口，本教程采用普通 Python cell。
已安装依赖后，主教程离线运行。Spatial 附录只展示后续写法，扩展需教师提前安装。

[官方 Jupyter 指南](https://duckdb.org/docs/lts/guides/python/jupyter)

In [ ]:
from pathlib import Path
from time import perf_counter
import gc
import os
import sys
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import duckdb
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.ticker import PercentFormatter
from IPython.display import display

assert (Path(sys.prefix) / 'conda-meta').exists(), '请在右上角选择 Conda 内核'
TUTORIAL_DIR = Path('.') if Path('data/posts_two_weeks.parquet').is_file() else Path('tutorials')
DATA = TUTORIAL_DIR / 'data/posts_two_weeks.parquet'
assert DATA.is_file(), '请从 tutorials 或项目根目录启动内核，并保留 tutorials/data'
ROOT = TUTORIAL_DIR.resolve().parent
OUT = ROOT / 'data/output/duckdb_tutorial'
OUT.mkdir(parents=True, exist_ok=True)
DEFAULT_THREADS = min(4, os.cpu_count() or 1)
RUN_FULL_MEMORY_DEMO = False  # 选讲：整表读入 Pandas，可能增加约 1 GiB 或更多进程内存。
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_columns', 8)
available_fonts = {font.name for font in font_manager.fontManager.ttflist}
CJK_FONT = next((name for name in ['Noto Sans CJK SC', 'Arial Unicode MS',
    'Microsoft YaHei', 'SimHei', 'Songti SC', 'Heiti TC', 'Noto Sans CJK JP']
    if name in available_fonts), None)
if CJK_FONT:
    plt.rcParams['font.family'] = CJK_FONT
plt.rcParams.update({'font.size': 11, 'axes.unicode_minus': False,
                     'axes.spines.top': False, 'axes.spines.right': False})

def label(chinese, english):
    return chinese if CJK_FONT else english

print('DuckDB：', duckdb.__version__, 'Pandas：', pd.__version__)
print('Python：', sys.executable)
print('本教程线程上限：', DEFAULT_THREADS)
print('唯一原始数据的相对路径：', DATA)

<a id="size"></a>
## 1．103 MiB 的文件，读入后还是 103 MiB 吗？

先读 Parquet 元数据确认规模，再用前课的四川范围演示 Pandas 的内存估算。
Parquet 按列存储并压缩；Pandas 中的字符串、索引与对象也占内存，所以文件体积与内存占用不能等同。
**行数、字段类型、文本长度和计算方法一起决定资源需求**。百万行并不自动意味着必须用集群。

`MiB = 字节 / 1024²`；`memory_usage(deep=True)` 尝试计入字符串等对象，
是表内存估算，不是整个 Python 进程的峰值，也不计入后续计算的全部临时对象。

In [ ]:
parquet = pq.ParquetFile(DATA)
file_info = pd.Series({
    '文件字节数': f'{DATA.stat().st_size:,}',
    '文件大小（MiB）': f'{DATA.stat().st_size / 1024**2:.2f}',
    '记录数': f'{parquet.metadata.num_rows:,}',
    '列数': str(parquet.metadata.num_columns),
    '行组数': str(parquet.metadata.num_row_groups),
}, name='实际规模')
display(file_info.to_frame())
sc_small = pd.read_parquet(DATA, columns=['city', 'user_id', 'text'],
    filters=[('province_code', '==', '51'), ('usable_text', '==', True)])
print('四川可用记录、用户数：', len(sc_small), sc_small['user_id'].nunique())
print('仅这三列的 Pandas 表内存估算（MiB）：',
      round(sc_small.memory_usage(deep=True).sum() / 1024**2, 2))

### 选讲：测量全国完整表的内存

开场的 `RUN_FULL_MEMORY_DEMO` 默认关闭。需要演示时改成 `True`，重新运行开场和这一格。
本机此前完整读取 20 列的深度内存估算约 784 MiB；不同版本和列类型可能得到不同估算。
此演示故意读取全部字段，便于解释体积差异；实际分析应先确定需要哪些列。
删除变量可释放对象引用，进程内存不保证立刻全部还给系统。

In [ ]:
full_memory_mib = None
if RUN_FULL_MEMORY_DEMO:
    full_demo = pd.read_parquet(DATA)
    full_memory_mib = full_demo.memory_usage(deep=True).sum() / 1024**2
    print('完整表行列数：', full_demo.shape)
    print('完整表内存估算（MiB）：', round(full_memory_mib, 2))
    print('估算与 Parquet 文件体积的比值：',
          round(full_memory_mib / (DATA.stat().st_size / 1024**2), 2))
    del full_demo
    gc.collect()
else:
    print('整表内存演示未启用；继续用 DuckDB 查询全国记录。')

<a id="connect"></a>
## 2．DuckDB 是什么？怎样直接查询文件？

DuckDB 是运行在当前 Python 进程内的分析型数据库引擎。
创建连接后，可以直接查询 Parquet，不需要先运行数据库服务器。
这里的 `:memory:` 表示数据库目录在内存中；源数据仍来自磁盘上的 Parquet，
并不表示要把整份文件全部复制进内存。

`posts` 是一个**视图**：保存“从哪里读数据”的查询定义。建立视图不会另复制一份帖子表。
真正执行查询时仍会消耗读取、解压和计算资源。
`con.execute(...).df()` 则执行 SQL，并把结果转成 Pandas；返回多少行需要由查询控制。

[Python API](https://duckdb.org/docs/lts/clients/python/overview)
与 [Parquet 视图](https://duckdb.org/docs/lts/data/parquet/overview)

In [ ]:
# 若重新运行此格，先关闭之前的连接。
if 'con' in globals():
    con.close()
con = duckdb.connect(database=':memory:', config={
    'threads': DEFAULT_THREADS,
    'memory_limit': '512MB',
    'temp_directory': str(OUT / 'tmp'),
    'max_temp_directory_size': '1GB',
    'autoinstall_known_extensions': False,
    'autoload_known_extensions': False,
})
con.read_parquet(str(DATA)).create_view('posts')
with pd.option_context('display.max_rows', 25):
    display(con.execute('DESCRIBE posts').df()[['column_name', 'column_type']])

### 核对全国规模与质量条件

`COUNT(*)` 数行；`COUNT(DISTINCT user_id)` 数不同的非空用户 ID。
`FILTER (WHERE ...)` 只限定它前面的聚合，便于把原始和可用计数放在同一行。
`usable_text` 是已有质量标记，沿用前课规则，不能把不可用记录混进内容比例的分母。

In [ ]:
overview = con.execute("""
    SELECT COUNT(*) AS n_raw,
           COUNT(*) FILTER (WHERE usable_text) AS n_usable,
           COUNT(DISTINCT user_id) FILTER (WHERE usable_text) AS n_usable_users,
           MIN(date) AS start_date, MAX(date) AS end_date,
           COUNT(DISTINCT date) AS n_dates
    FROM posts
""").df()
display(overview)
assert int(overview.loc[0, 'n_raw']) == parquet.metadata.num_rows

**结果解读：** 输出仍只有一行。1,254,689 是原始行数，1,149,693 是可用记录数；
用户数需要在选定质量范围内去重。这里的用户是已观测匿名用户，不是人口规模。
`date` 源字段采用统一的 `YYYY-MM-DD` 字符串，本例可用于核对两周窗口。

<a id="sql"></a>
## 3．SQL 入门：全国哪些城市标签的记录更多？

从前面的 Pandas 操作理解 SQL：

| SQL | 做什么 | Pandas 中熟悉的操作 |
|---|---|---|
| `FROM posts` | 确定输入 | 选择 DataFrame |
| `WHERE usable_text` | 筛选可用记录 | 布尔条件筛选 |
| `GROUP BY city` | 每个城市一组 | `groupby('city')` |
| `COUNT(*) AS n_posts` | 每组数行并命名 | `size()` |
| `ORDER BY n_posts DESC, city` | 按数量降序，城市名处理并列 | `sort_values` |
| `LIMIT 5` | 限制输出行数 | `head(5)` |

阅读时可按“从哪里读 → 保留哪些行 → 怎样分组 → 算什么 → 排序”理解。
我们先保留完整城市汇总表，再用 `head(10)` 显示前十项，方便后面核对总量。

In [ ]:
city_counts_sql = """
    SELECT city, COUNT(*) AS n_posts
    FROM posts
    WHERE usable_text
    GROUP BY city
    ORDER BY n_posts DESC, city
"""
national_counts = con.execute(city_counts_sql).df()
display(national_counts.head(10))
print('全国城市汇总行数：', len(national_counts))
print('汇总记录数：', national_counts['n_posts'].sum())
print('汇总表内存估算（KiB）：', round(national_counts.memory_usage(deep=True).sum() / 1024, 2))

**结果解读：** 百万条输入被聚合成每城市一行。城市标签按源字段解释，
数量差异也可能来自采集覆盖；不能据此给城市居民的活跃程度排名。
源数据的 `Unknown` 表示城市未知，不是一座城市：汇总表保留它以核对全部记录，
后面的城市排名图会排除它，并单独报告其记录数。
`LIMIT` 限制的是最终输出；含排序、分组时，返回五行也可能需要先处理大量输入。

### 同一个文件，如何只分析四川？

`?` 是参数占位符，参数列表按出现顺序传入。省编码是字符串，四川为 `'51'`。
改 `PROVINCE_CODE` 并重跑本格即可查看其他省编码。
参数用于传入值；本例不将它用作表名或列名。

In [ ]:
PROVINCE_CODE = '51'
province_overview = con.execute("""
    SELECT COUNT(*) AS n_posts, COUNT(DISTINCT user_id) AS n_users
    FROM posts
    WHERE usable_text AND province_code = ?
""", [PROVINCE_CODE]).df()
display(province_overview)
print('省编码：', PROVINCE_CODE)

<a id="metrics"></a>
## 4．从记录量到用户与内容指标

### 4.1 全国城市表：记录数、不同用户数与“工作”关键词比例

这张表的一行是一个城市标签：

- `n_posts`：可用帖子记录数。
- `n_users`：该城市标签内的不同匿名用户数。
- `n_work`：正文含指定关键词的记录数；`contains` 做字面匹配。
- `work_share`：命中记录数 / 该城市全部可用记录数。
- `posts_per_user`：记录数 / 该城市不同用户数。

`WITH counts AS (...)` 将中间汇总命名，便于下一步计算比例；它本身不要求保存一张磁盘表。
`COALESCE(text, '')` 把缺失正文视为空字符串；`NULLIF(分母, 0)` 避免除零。
同一用户可能出现于多个城市标签，所以各城市用户数相加不一定等于全国去重用户数。

In [ ]:
KEYWORD = '工作'
city_metrics_sql = """
    WITH counts AS (
        SELECT city,
               COUNT(*) AS n_posts,
               COUNT(DISTINCT user_id) AS n_users,
               COUNT(*) FILTER (WHERE contains(COALESCE(text, ''), ?)) AS n_work
        FROM posts
        WHERE usable_text
        GROUP BY city
    )
    SELECT city, n_posts, n_users, n_work,
           n_work * 1.0 / NULLIF(n_posts, 0) AS work_share,
           n_posts * 1.0 / NULLIF(n_users, 0) AS posts_per_user
    FROM counts
    ORDER BY n_posts DESC, city
"""
national_city = con.execute(city_metrics_sql, [KEYWORD]).df()
display(national_city.head(10))
print('关键词：', KEYWORD)
print('可用记录中的全国不同用户数：', int(overview.loc[0, 'n_usable_users']))
print('各城市用户数之和：', int(national_city['n_users'].sum()))

**结果解读：** 关键词比例描述文字命中，不表示职业、工作状态、偏好或心理状态。
`posts_per_user` 是采集记录的平均数，不能替代中位数，也不能当成完整历史的发帖频率。
更换关键词后，`work_share` 的计算含义随之改变；讲解时也要同步修改图题。

### 4.2 四川：DuckDB 与前课 Pandas 是否得到同一个结果？

这里固定核对四川可用记录，与开场 `sc_small` 的范围一致。
Pandas 已通过 `columns` 和 `filters` 限定读取，所以比较的是两种合理写法。
不用不同质量条件、不同统计单位的结果比较“谁更快”。

In [ ]:
sc_sql = """
    SELECT city, COUNT(*) AS n_posts, COUNT(DISTINCT user_id) AS n_users,
           COUNT(*) FILTER (WHERE contains(COALESCE(text, ''), ?)) AS n_work
    FROM posts
    WHERE usable_text AND province_code = '51'
    GROUP BY city
    ORDER BY city
"""
sc_duck = con.execute(sc_sql, [KEYWORD]).df()
sc_pandas = sc_small.assign(has_work=sc_small['text'].str.contains(KEYWORD, regex=False, na=False))
sc_pandas = (sc_pandas.groupby('city', dropna=False)
    .agg(n_posts=('user_id', 'size'), n_users=('user_id', 'nunique'), n_work=('has_work', 'sum'))
    .reset_index().sort_values('city').reset_index(drop=True))
pd.testing.assert_frame_equal(sc_duck, sc_pandas, check_dtype=False)
display(sc_duck.head(6))
print('核对通过：各城市记录数、不同用户数和关键词命中数完全一致。')

<a id="plot"></a>
## 5．汇总后绘图：百万条输入，十行画图数据

DuckDB 已完成筛选、分组和计数；Pandas 只承接每城市一行的结果。
下面排除城市未知的标签，再取记录最多的十个城市，在同一顺序下比较记录数和关键词比例。
这是按记录量选择的十个城市，**不是关键词比例最高的十个城市**。

如果 `.df()` 接收的是 `SELECT * FROM posts`，仍会把大量原始数据转入 Pandas，
因此关键是先确定最终需要的结果规模。

In [ ]:
unknown_city = national_city.loc[national_city['city'].isna() | national_city['city'].eq('Unknown')]
known_city = national_city.loc[national_city['city'].notna() & national_city['city'].ne('Unknown')]
top10 = known_city.head(10).sort_values(['n_posts', 'city'])
print('城市未知的可用记录数：', int(unknown_city['n_posts'].sum()), '；只从排名图中排除，保留在总量中。')
fig_city, axes_city = plt.subplots(1, 2, figsize=(12, 5), sharey=True, layout='constrained')
bars = axes_city[0].barh(top10['city'], top10['n_posts'], color='#2878B5')
axes_city[0].bar_label(bars, fmt='%d', padding=3, fontsize=9)
axes_city[0].set(title=label('全国记录量前十城市标签', 'Ten city labels with most records'),
    xlabel=label('可用帖子记录数（条）', 'Usable post records'),
    ylabel=label('源数据城市标签', 'Source city label'),
    xlim=(0, top10['n_posts'].max() * 1.25))
axes_city[1].barh(top10['city'], top10['work_share'], color='#E07A35')
axes_city[1].set(title=label(f'同一组城市：“{KEYWORD}”关键词比例', f'Same cities: keyword "{KEYWORD}" share'),
    xlabel=label('占本城市可用记录的比例', 'Share of usable city records'))
axes_city[1].yaxis.set_tick_params(labelleft=False)
axes_city[1].xaxis.set_major_formatter(PercentFormatter(xmax=1))
for ax in axes_city:
    ax.grid(axis='x', alpha=0.25)
plt.show()
print('送入画图的城市数：', len(top10), '；各城市比例的分母为该城市可用记录数。')

<a id="plan"></a>
## 6．查询计划：引擎实际准备做哪些工作？

### 6.1 列裁剪与过滤下推

下面只查一个源文件日期的城市记录数。`source_date` 是采集源文件日期，
此处用于讲文件扫描；不要未经核对就将它当作发帖时间。
`EXPLAIN` 只显示计划，不运行查询；`EXPLAIN ANALYZE` 会真正运行并显示执行统计。

观察 `READ_PARQUET` 或 `TABLE_SCAN` 区域：`Projections` 是需要输出的列，
`Filters` 是交给读取阶段处理的条件；过滤所需列也必须读取。
无需参与此查询的 `text`、位置和情绪分数字段可以跳过。
文件内行组的统计范围合适时，还可能跳过整块数据，效果取决于数据布局。

[Parquet 部分读取](https://duckdb.org/docs/lts/data/parquet/overview)
与 [查询性能指南](https://duckdb.org/docs/lts/guides/performance/how_to_tune_workloads)

In [ ]:
plan_sql = """
    SELECT city, COUNT(*) AS n_posts
    FROM posts
    WHERE usable_text AND source_date = ?
    GROUP BY city
    ORDER BY n_posts DESC, city
"""
SOURCE_DAY = '2022-08-12'
plan_text = con.execute('EXPLAIN ' + plan_sql, [SOURCE_DAY]).fetchone()[1]
print(plan_text)

In [ ]:
analyzed_plan = con.execute('EXPLAIN ANALYZE ' + plan_sql, [SOURCE_DAY]).fetchone()[1]
print(analyzed_plan)
day_city = con.execute(plan_sql, [SOURCE_DAY]).df()
print('该源文件日期的可用记录总数：', int(day_city['n_posts'].sum()))

### 6.2 看看这一个文件怎样分成行组

**行组（row group）不是另一份数据文件**，是 Parquet 内部的存储块。
下面只读元数据，展示每个块的行数及源日期最小/最大值。
若某一块的日期范围完全不满足条件，引擎有机会跳过该块。
计划中的 `~... rows` 是估算行数，不能用它替代真实记录数；实际计数以查询结果为准。

In [ ]:
source_date_column = parquet.schema_arrow.names.index('source_date')
row_groups = []
for i in range(parquet.metadata.num_row_groups):
    group = parquet.metadata.row_group(i)
    stats = group.column(source_date_column).statistics
    row_groups.append({
        'row_group': i, 'n_rows': group.num_rows,
        'min_source_date': stats.min if stats and stats.has_min_max else None,
        'max_source_date': stats.max if stats and stats.has_min_max else None,
    })
row_group_info = pd.DataFrame(row_groups)
with pd.option_context('display.max_rows', 20):
    display(row_group_info)
possible = row_group_info.loc[
    row_group_info['min_source_date'].isna()
    | ((row_group_info['min_source_date'] <= SOURCE_DAY)
       & (row_group_info['max_source_date'] >= SOURCE_DAY))]
print('仅按日期范围推算的候选行组数：', len(possible), '/', len(row_group_info))
print('这是元数据推算，不是磁盘读取字节数或实际跳过块数的测量。')

<a id="resources"></a>
## 7．多核与内存：数据继续增大怎么办？

### 7.1 多个线程一定比一个线程快吗？

DuckDB 可以在一台电脑内并行执行部分查询。这里对同一全国城市指标查询做线程数对比：
每种线程数先运行一次预热，再交替运行三轮，计时包含执行和将小结果表转成 Pandas。
每次都核对结果，报告中位数及范围。

这是**当前机器、当前查询、缓存已预热**的课堂观察，不是跨库或冷磁盘基准。
多核收益取决于扫描规模、存储布局、磁盘、计算量与调度成本；一次结果不能证明普遍加速。
查询里的 `COUNT(DISTINCT ...)` 和关键词匹配也需要计算，工作量不同于只查元数据行数。

In [ ]:
thread_options = sorted({1, DEFAULT_THREADS})
timing_records = []
try:
    for n_threads in thread_options:
        con.execute(f'SET threads = {n_threads}')
        warm_result = con.execute(city_metrics_sql, [KEYWORD]).df()
        pd.testing.assert_frame_equal(warm_result, national_city, check_dtype=False)
    for round_number in range(1, 4):
        order = thread_options if round_number % 2 else list(reversed(thread_options))
        for n_threads in order:
            con.execute(f'SET threads = {n_threads}')
            start = perf_counter()
            measured = con.execute(city_metrics_sql, [KEYWORD]).df()
            elapsed_ms = (perf_counter() - start) * 1000
            pd.testing.assert_frame_equal(measured, national_city, check_dtype=False)
            timing_records.append({'threads': n_threads, 'round': round_number, 'elapsed_ms': elapsed_ms})
finally:
    con.execute(f'SET threads = {DEFAULT_THREADS}')
timings = pd.DataFrame(timing_records)
timing_summary = timings.groupby('threads')['elapsed_ms'].agg(['median', 'min', 'max'])
display(timing_summary.rename(columns={'median': '中位数（ms）', 'min': '最小（ms）', 'max': '最大（ms）'}).round(2))
if len(thread_options) == 1:
    print('本环境只报告一个 CPU，未做不同线程数对比。')
else:
    print('当前查询的中位耗时比（1线程 / 多线程）：',
          round(timing_summary.loc[1, 'median'] / timing_summary.loc[DEFAULT_THREADS, 'median'], 2))

### 7.2 内存限制和磁盘临时空间

开场连接设置了 `memory_limit='512MB'`、线程上限和课程输出目录内的临时路径。
部分排序、分组和连接在内存不足时，可以将中间数据暂存磁盘，称为**溢写（spill）**。
用磁盘换内存需要额外 I/O、空间和时间，也有算子限制；不能保证所有查询都能在任意小内存中运行。
`memory_limit` 管理的内存也不等于整个 Python 进程的硬上限，Pandas 结果等另占内存。

当前样例不一定触发溢写。下方只查看设置，不把设置成功当成超内存计算的验证。
`temp_directory` 是临时计算路径，不是新增原始数据集；上限 1GB 指临时磁盘空间的配置值。

In [ ]:
settings = con.execute("""
    SELECT name, value
    FROM duckdb_settings()
    WHERE name IN ('threads', 'memory_limit', 'temp_directory', 'max_temp_directory_size')
    ORDER BY name
""").df()
display(settings)

### 7.3 将两周扩展到全年时，先检查什么？

1. **字段和质量口径一致吗？** 先明确日期、用户去重和可用文本条件。
2. **查询需要原始全文吗？** 城市计数无需读正文，关键词比例才需要正文。
3. **能否先减少结果？** 在引擎内筛选和聚合，再将小表交给 Pandas。
4. **数据怎样组织？** 更大的项目可按常用筛选字段组织 Parquet；读取多文件时仍可用同一 SQL 思路。
5. **瓶颈在哪里？** 用执行计划和实际耗时检查，再调整线程、内存与临时磁盘空间。

本课保留现有单一源文件，不为演示复制或重复记录来冒充更大的真实样本。
如果进行按块统计，记录数等计数容易合并；跨块不同用户数不能把各块去重数简单相加。
**原始大表交给 DuckDB 计算，最终分析结果交给 Pandas 整理和绘图。**

<a id="spatial"></a>
## 附录：DuckDB Spatial 如何衔接第二次课？

第二次课仍使用同一个文件的 `longitude_grid`、`latitude_grid` 和 `coordinate_valid`。
`spatial` 扩展提供几何创建、空间关系、距离与空间文件读取等能力。
本节介绍入口，正式的时段、位置质量和时空行为分析在第二次课完成。

| 函数入口 | 后续能做什么 | 使用前需要明确 |
|---|---|---|
| `ST_Point(x, y)` | 从坐标生成点 | 横纵坐标顺序、坐标来源 |
| `ST_Intersects(a, b)` | 判断几何是否相交 | 几何对象必须采用兼容坐标体系 |
| `ST_Distance(a, b)` | 计算平面几何距离 | 结果单位来自坐标；经纬度直接计算不能称为米 |
| `ST_Read(...)` | 读取空间文件 | 格式支持、坐标体系与字段 |

### 教师提前准备的写法（下方是说明，不参与本教程运行）

扩展首次安装需要可用网络与匹配的 DuckDB 版本、平台；安装后在连接中显式加载。
`INSTALL` 只准备扩展，`LOAD` 使当前连接可以调用它。

```python
con.execute('INSTALL spatial')  # 教师课前执行一次
con.execute('LOAD spatial')
spatial_preview = con.execute("""
    SELECT ST_AsText(ST_Point(longitude_grid, latitude_grid)) AS point_wkt
    FROM posts
    WHERE usable_text AND coordinate_valid
      AND province_code = '51'
      AND longitude_grid IS NOT NULL AND latitude_grid IS NOT NULL
    LIMIT 3
""").df()
display(spatial_preview)
```

这些点来自源数据舍入后的 0.02 度网格，不是精确个人位置；
`coordinate_valid` 只是粗范围质量标记，坐标体系和定位机制尚未核实。
创建几何对象不会解决这些不确定性，不能立即据此计算通勤距离或恢复路线。
确认坐标体系之后，再设计行政区空间连接和距离分析。

[Spatial 扩展安装与加载](https://duckdb.org/docs/lts/core_extensions/spatial/overview)
与 [空间函数参考](https://duckdb.org/docs/lts/core_extensions/spatial/functions)

<a id="reference"></a>
## 速查与参考资料

| 目的 | 本教程入口 |
|---|---|
| 直接指向 Parquet | `con.read_parquet(str(DATA)).create_view('posts')` |
| 运行 SQL 并接收小表 | `con.execute(sql, parameters).df()` |
| 数记录 / 数不同用户 | `COUNT(*)` / `COUNT(DISTINCT user_id)` |
| 筛选可用范围 | `WHERE usable_text AND province_code = ?` |
| 关键词字面匹配 | `contains(COALESCE(text, ''), ?)` |
| 看计划 / 实际执行统计 | `EXPLAIN` / `EXPLAIN ANALYZE` |
| 控制线程数 | `SET threads = 1` |
| 控制计算资源 | 连接配置 `memory_limit`、`temp_directory` |

- 用户指定的 [Jupyter 官方指南](https://duckdb.org/docs/lts/guides/python/jupyter)：原生接口与可选 SQL 单元。
- 用户指定的 [JupySQL Plotting with DuckDB](https://duckdb.org/2023/02/24/jupysql)：借鉴先聚合再绘图的思路，
  课堂实现使用现有 Matplotlib，不新增 JupySQL 依赖。
- [Python API](https://duckdb.org/docs/lts/clients/python/overview) 与
  [参数化查询](https://duckdb.org/docs/lts/clients/python/dbapi)。
- [Parquet 读取](https://duckdb.org/docs/lts/data/parquet/overview) 与
  [性能及超内存处理](https://duckdb.org/docs/lts/guides/performance/how_to_tune_workloads)。

主教程按 DuckDB 1.4 LTS 接口组织，版本在开场打印。官网页面可能更新，实际结果以当前运行环境为准。
主教程使用真实单一源，核心统计核对全国总量和四川 Pandas 结果。
默认关闭的整表内存演示不属于默认执行；Spatial 为说明代码，不把扩展写法介绍计为实际空间计算。

### 收尾：检查统计口径并释放连接

下面核对城市计数与总量、关键词比例范围，以及四川的统一范围。
关闭连接后若要重跑 SQL，先重跑第 2 部分的连接单元。

In [ ]:
pd.testing.assert_frame_equal(national_counts, national_city[['city', 'n_posts']], check_dtype=False)
assert national_city['n_posts'].sum() == int(overview.loc[0, 'n_usable'])
assert national_city['n_work'].between(0, national_city['n_posts']).all()
assert national_city['work_share'].between(0, 1).all()
np.testing.assert_allclose(national_city['work_share'], national_city['n_work'] / national_city['n_posts'])
np.testing.assert_allclose(national_city['posts_per_user'], national_city['n_posts'] / national_city['n_users'])
assert sc_duck['n_posts'].sum() == len(sc_small) == 54609
assert sc_small['user_id'].nunique() == 36463
assert known_city['n_posts'].sum() + unknown_city['n_posts'].sum() == int(overview.loc[0, 'n_usable'])
assert top10['city'].notna().all() and top10['city'].ne('Unknown').all()
assert row_group_info['n_rows'].sum() == parquet.metadata.num_rows
print('全国总量、指标分母和四川 Pandas 对照核对通过；本课只使用一份原始 Parquet。')
con.close()
parquet.close()